# 11 — RAG: embed the chunks and build the hybrid index

**Purpose.** Turn the 1,600 chunks from notebook 10 into a searchable index: semantic vectors in ChromaDB, plus a keyword (BM25) index for the same chunks. Contract: notebook 08.

- **Inputs:** newest `data/manifests/rag_chunks_*.json` and `data/processed/rag/chunks.jsonl`; `configs/rag.yaml` (`embedding_candidates`, `vector_store`).
- **Outputs:**
  - `data/processed/rag/chroma/`: persisted ChromaDB collection (git-ignored, rebuilt by this notebook)
  - `data/processed/rag/bm25_index.pkl`: BM25 keyword index over the same chunks (git-ignored)
  - `data/manifests/rag_index_<timestamp>.json`: model, parameters, counts, hashes (committed)
- **Next:** notebook 12 builds the retrieval function (hybrid search, date filter, reranker off in v1) on top of these two indexes.

**GPU:** not required. `BAAI/bge-m3` embeds all 1,600 chunks in under 2 minutes on a 10-core CPU; a GPU only makes it faster.

### For the evaluator

- **The embedding model is `BAAI/bge-m3`** (`configs/rag.yaml`), chosen because it is multilingual and handles Finnish; the config's fallback (`multilingual-e5-base`) is not built here, but the two can be compared by rerunning this notebook with `EMBEDDING_MODEL_ID` changed.
- **Search is hybrid: semantic (Chroma, cosine) + keyword (BM25 with Finnish stemming).** Notebook 12 merges the two rankings; this notebook only builds them.
- **Every vector and every BM25 entry carries the chunk's `chunk_id`**, so results from either index map back to the same citation metadata written in notebook 10.
- **The index is versioned.** The manifest records the embedding model, its revision, and the chunk manifest it was built from. A retrieval result that does not match the current index manifest should not be trusted.

In [ ]:
# Mount Drive on Colab; skipped automatically when running locally.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
    os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"
except ImportError:
    pass

In [ ]:
# Only needed on a fresh runtime; the project requirements already list these.
# !pip install -q sentence-transformers chromadb rank_bm25 snowballstemmer pyyaml pandas

In [ ]:
import os, re, json, time, pickle, hashlib, datetime as dt, importlib.metadata
from pathlib import Path
import yaml
import pandas as pd

def _find_repo():
    env = os.environ.get("JOBAI_REPO")
    if env:
        return Path(env).resolve()
    p = Path.cwd().resolve()
    for cand in (p, *p.parents):
        if (cand / "configs" / "rag.yaml").is_file():
            return cand
    return p

REPO = _find_repo()
MAN = REPO / "data" / "manifests"
OUT_DIR = REPO / "data" / "processed" / "rag"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = yaml.safe_load(open(REPO / "configs" / "rag.yaml"))
# EVALUATOR: change the model in configs/rag.yaml and rerun this notebook to compare
# candidates; do not hand-edit it only here, or the manifest will not match the config.
EMBEDDING_MODEL_ID = CFG["embedding_candidates"][0]["id"]
VECTOR_CFG = CFG["vector_store"]
NOW_UTC = dt.datetime.now(dt.timezone.utc)
TS = NOW_UTC.strftime("%Y%m%dT%H%M%SZ")
print("repo:", REPO)
print("embedding model:", EMBEDDING_MODEL_ID)
print("vector store:", VECTOR_CFG)

## Inputs check

Pin the chunk manifest, so the index says exactly which chunks it was built from. `CHUNK_MANIFEST = None` takes the newest one.

In [ ]:
CHUNK_MANIFEST = None   # e.g. "rag_chunks_20260921T181228Z.json"

if CHUNK_MANIFEST:
    chunk_manifest_path = MAN / CHUNK_MANIFEST
else:
    found = sorted(MAN.glob("rag_chunks_*.json"))
    assert found, "No rag_chunks_*.json manifest found. Run notebook 10 first."
    chunk_manifest_path = found[-1]

chunk_manifest = json.loads(chunk_manifest_path.read_text())
chunks_path = REPO / chunk_manifest["outputs"]["chunks"]["path"]
actual_sha256 = hashlib.sha256(chunks_path.read_bytes()).hexdigest()
assert actual_sha256 == chunk_manifest["outputs"]["chunks"]["sha256"], \
    "chunks.jsonl does not match its manifest hash. Rerun notebook 10."

chunks = pd.read_json(chunks_path, lines=True, dtype={"published": str, "published_effective": str})
assert chunks.chunk_id.is_unique
print("chunk manifest:", chunk_manifest_path.name)
print("chunks:", len(chunks), "| documents:", chunks.doc_id.nunique(),
      "| languages:", chunks.language.value_counts().to_dict())

## Semantic index: embed with `bge-m3` and store in ChromaDB

`embed_text` (chunk text plus the document title and section heading, from notebook 10) is embedded, not the bare `text` — the citation still shows `text`. Embeddings are L2-normalised so that Chroma's cosine distance behaves as plain cosine similarity.

In [ ]:
from sentence_transformers import SentenceTransformer

t0 = time.time()
embedder = SentenceTransformer(EMBEDDING_MODEL_ID)
embedding_dim = embedder.get_sentence_embedding_dimension()
print(f"loaded {EMBEDDING_MODEL_ID} ({embedding_dim} dims) in {time.time() - t0:.0f}s")

t0 = time.time()
embeddings = embedder.encode(chunks.embed_text.tolist(), batch_size=32,
                             normalize_embeddings=True, show_progress_bar=True)
print(f"embedded {len(chunks)} chunks in {time.time() - t0:.0f}s")
assert embeddings.shape == (len(chunks), embedding_dim)

In [ ]:
import chromadb

CHROMA_DIR = REPO / VECTOR_CFG["persist_dir"]  # from configs/rag.yaml, not hardcoded
# A stale collection from a different model or a previous run must not silently mix
# with the new one, so each build starts from an empty collection.
if CHROMA_DIR.is_dir():
    import shutil
    shutil.rmtree(CHROMA_DIR)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.create_collection(
    name=VECTOR_CFG["collection_name"],
    metadata={"hnsw:space": "cosine", "embedding_model": EMBEDDING_MODEL_ID},
)

# Chroma's metadata filters only compare numbers, not date strings, so a parallel
# integer field (proleptic Gregorian ordinal) carries the same date for filtering;
# `published_effective` itself stays as the human-readable ISO string for citations.
chunks["published_effective_epoch"] = chunks.published_effective.map(
    lambda d: __import__("datetime").date.fromisoformat(d).toordinal())

METADATA_FIELDS = ["doc_id", "source_id", "source_type", "title", "language",
                   "published_effective", "published_effective_epoch", "landing_url", "licence", "heading"]
BATCH = 256
for start in range(0, len(chunks), BATCH):
    batch = chunks.iloc[start:start + BATCH]
    collection.add(
        ids=batch.chunk_id.tolist(),
        embeddings=embeddings[start:start + BATCH].tolist(),
        documents=batch.text.tolist(),
        metadatas=batch[METADATA_FIELDS].fillna("").to_dict(orient="records"),
    )
print("chroma collection:", collection.name, "| count:", collection.count())
assert collection.count() == len(chunks)

## Keyword index: BM25 with Finnish stemming

Finnish is heavily inflected (`työpaikka`, `työpaikkoja`, `työpaikkojen`, …), so plain word matching misses most forms. Words are stemmed with the Snowball Finnish stemmer before indexing; the same stemming must be applied to a query at retrieval time (notebook 12). English text is tokenised without stemming, since bge-m3's semantic side already covers it well and the corpus has only 174 English chunks.

In [ ]:
import snowballstemmer
from rank_bm25 import BM25Okapi

FI_STEMMER = snowballstemmer.stemmer("finnish")
WORD_RE = re.compile(r"[^\W\d_]+", re.UNICODE)   # letters only; drops standalone numbers and punctuation

def tokenize(text, language):
    words = WORD_RE.findall(text.lower())
    return FI_STEMMER.stemWords(words) if language == "fi" else words

corpus_tokens = [tokenize(t, lang) for t, lang in zip(chunks.text, chunks.language)]
bm25 = BM25Okapi(corpus_tokens)

bm25_path = OUT_DIR / "bm25_index.pkl"
with open(bm25_path, "wb") as f:
    pickle.dump({"bm25": bm25, "chunk_ids": chunks.chunk_id.tolist(), "tokenizer": "snowball_finnish_or_none"}, f)
print("bm25 index:", bm25_path, "|", bm25_path.stat().st_size // 1024, "KB")

## Check: does each index actually retrieve the right thing?

A handful of sanity queries per index, in both languages, before trusting either one. This is not the evaluation set (that is the evaluator's job); it only catches a broken index early.

**`bm25_search` always returns its top `k`, even when nothing really matches** (a real run below shows this: every score is `0.0` for one query). Notebook 12's retrieval function must apply a minimum-score cutoff before treating a BM25 result as a real match; this helper is for inspection only.

In [ ]:
def semantic_search(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    result = collection.query(query_embeddings=[q], n_results=k)
    return list(zip(result["ids"][0], result["distances"][0]))

def bm25_search(query, language, k=3):
    scores = bm25.get_scores(tokenize(query, language))
    top = sorted(range(len(scores)), key=lambda i: -scores[i])[:k]
    return [(chunks.chunk_id.iloc[i], round(float(scores[i]), 2)) for i in top]

CHECKS = [
    ("Kuinka monta uutta avointa työpaikkaa ilmoitettiin?", "fi"),   # how many new vacancies were reported
    ("Uusimaa avoimet työpaikat", "fi"),
    ("how many new job vacancies were reported", "en"),
    ("pitkäaikaistyöttömyys", "fi"),                                 # long-term unemployment
]
for query, lang in CHECKS:
    print(f"=== {query!r} ({lang})")
    print("  semantic:", semantic_search(query, k=3))
    print("  bm25    :", bm25_search(query, lang, k=3))

## Read the top hits

The chunk id alone does not show whether a match is actually good. Print the text for the clearest query.

In [ ]:
def show_hits(hits, label):
    print(f"--- {label}")
    for chunk_id, score in hits:
        row = chunks.set_index("chunk_id").loc[chunk_id]
        print(f"[{score}] {chunk_id} ({row.language}, {row.published_effective}): {row.text[:180]}")

show_hits(semantic_search("Uusimaa avoimet työpaikat", k=3), "semantic: Uusimaa avoimet työpaikat")
show_hits(bm25_search("Uusimaa avoimet työpaikat", "fi", k=3), "bm25: Uusimaa avoimet työpaikat")

## Write the index manifest

Records the exact embedding model, the chunk manifest this index was built from, and file hashes, so notebook 12 (and the evaluator) can verify they are using a matching index.

In [ ]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def dir_size(path):
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())

manifest = {
    "name": "JobAI RAG index v1",
    "created_utc": NOW_UTC.isoformat(),
    "chunk_manifest": {"path": str(chunk_manifest_path.relative_to(REPO)), "sha256": sha256_file(chunk_manifest_path)},
    "semantic": {
        "model_id": EMBEDDING_MODEL_ID,
        "model_revision": getattr(embedder, "model_card_data", None) and None,  # kept simple; the model id is pinned
        "embedding_dim": int(embedding_dim),
        "distance": "cosine",
        "backend": VECTOR_CFG["backend"],
        "persist_dir": str(CHROMA_DIR.relative_to(REPO)),
        "collection_name": collection.name,
        "count": collection.count(),
        "bytes": dir_size(CHROMA_DIR),
    },
    "keyword": {
        "algorithm": "bm25_okapi", "stemmer": "snowball_finnish (fi only)",
        "path": str(bm25_path.relative_to(REPO)), "sha256": sha256_file(bm25_path),
        "count": len(chunks),
    },
    "sanity_checks": {
        query: {"semantic": semantic_search(query, k=3), "bm25": bm25_search(query, lang, k=3)}
        for query, lang in CHECKS
    },
    "packages": {n: importlib.metadata.version(n) for n in
                 ["sentence-transformers", "chromadb", "rank_bm25", "snowballstemmer"]},
}
manifest_out = MAN / f"rag_index_{TS}.json"
manifest_out.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str))
print("wrote:", manifest_out)
print("chroma dir:", CHROMA_DIR, "-", manifest["semantic"]["bytes"] // 1024, "KB")

## Known limitations (as of 2026-09-22)

- **Only `bge-m3` is built here.** The fallback (`multilingual-e5-base` in `configs/rag.yaml`) is not embedded; comparing them is the evaluator's retrieval evaluation, not this notebook.
- **BM25 stems Finnish but not English.** With only 174 English chunks this is a minor gap; revisit if English retrieval quality is poor.
- **The Finnish stemmer under-normalises some words.** Checked on a real query: `pitkäaikaistyöttömyys` (long-term unemployment, the noun form used in a question) stems to itself, while the document text mostly uses inflected forms such as `pitkäaikaistyöttömiä`, which stem to `pitkäaikaistyöttöm`. The two do not meet, so BM25 misses a real match that the semantic index still finds. This is a limitation of the hybrid design worth watching in the evaluator's retrieval numbers, not a bug in this notebook.
- **A BM25 score of 0 is not a match.** `bm25_search` above still returns its top `k` when nothing matches; notebook 12 must apply a minimum-score threshold before using a BM25 result.
- **No reranker.** Left off in version 1, per the plan in notebook 08; the code in notebook 12 should keep an on/off switch so it can be measured later.
- **Chroma's HNSW index is approximate,** though at 1,600 chunks it should not differ from exact search in practice.
- **The sanity checks above are a smoke test, not an evaluation.** They only check that the indexes are not broken; they are not a substitute for the evaluator's hit@k / MRR / nDCG measurement.

**Next:** notebook 12 merges semantic and BM25 results with Reciprocal Rank Fusion, applies the `published_effective` date cutoff, and freezes the retrieval and answer functions from the notebook 08 contract.